# LSTM



<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/auduvignac/Deep-learning/blob/main/keras-basic/lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

In [1]:
!wget https://www.lamsade.dauphine.fr/%7Ecazenave/sinwave.csv

--2025-03-24 19:19:17--  https://www.lamsade.dauphine.fr/%7Ecazenave/sinwave.csv
Resolving www.lamsade.dauphine.fr (www.lamsade.dauphine.fr)... 193.48.71.250
Connecting to www.lamsade.dauphine.fr (www.lamsade.dauphine.fr)|193.48.71.250|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 61711 (60K) [text/csv]
Saving to: ‘sinwave.csv’

sinwave.csv         100%[===================>]  60.26K   252KB/s    in 0.2s    

2025-03-24 19:19:18 (252 KB/s) - ‘sinwave.csv’ saved [61711/61711]



In [2]:
filename = "sinwave.csv"
f = open(filename, 'rb').read()
data = f.decode().split('\n')

In [3]:
print(data)

['0.841470985', '0.873736397', '0.90255357', '0.927808777', '0.949402346', '0.967249058', '0.98127848', '0.991435244', '0.997679266', '0.999985904', '0.998346054', '0.992766189', '0.983268329', '0.969889958', '0.952683874', '0.931717983', '0.907075026', '0.878852258', '0.847161063', '0.812126509', '0.773886863', '0.73259304', '0.688408006', '0.64150614', '0.592072543', '0.540302306', '0.486399742', '0.430577581', '0.373056127', '0.314062391', '0.253829194', '0.192594249', '0.130599223', '0.068088781', '0.005309624', '-0.057490488', '-0.120063711', '-0.182163097', '-0.243543569', '-0.303962886', '-0.3631826', '-0.420968998', '-0.477094024', '-0.531336178', '-0.583481391', '-0.633323869', '-0.680666907', '-0.725323664', '-0.7671179', '-0.805884672', '-0.841470985', '-0.873736397', '-0.90255357', '-0.927808777', '-0.949402346', '-0.967249058', '-0.98127848', '-0.991435244', '-0.997679266', '-0.999985904', '-0.998346054', '-0.992766189', '-0.983268329', '-0.969889958', '-0.952683874', '-0.

In [9]:
import numpy as np
# lecture des donnees et découpage en sequence de seq_len elements
def load_data(filename, seq_len):
  f = open(filename, 'rb').read()
  data = np.array(f.decode().split('\n'), dtype = np.float32)
  sequence_length = seq_len + 1
  result = []
  for index in range(len(data) - sequence_length):
    result.append(data[index: index + sequence_length])
  result = np.array(result)
  np.random.shuffle(result)
  row = round(0.9 * result.shape[0])
  train = result[:int(row), :] # 90 % des exemples pour l’apprentissage
  x_train = train[:, :-1] # On prend les séquences jusqu’à l’avant dernier élément
  y_train = train[:, -1] # On prend le dernier élément comme sortie à apprendre
  x_test = result[int(row):, :-1]
  y_test = result[int(row):, -1]
  # on transforme en un tenseur de dimension 3 avec une seule feature
  x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))
  x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1))
  return x_train, y_train, x_test, y_test

In [11]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,LSTM,Dropout
model = Sequential()
model.add(LSTM(50, input_shape=(50, 1), return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100, return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(1))
model.compile(loss="mse", optimizer="rmsprop", metrics = ['mae'])
X_train, y_train, X_test, y_test = load_data(filename, 50)
model.fit(X_train, y_train, batch_size=512, epochs=1, validation_split=0.05)
predict = model.predict (X_test)

9/9 ━━━━━━━━━━━━━━━━━━━━ 7s 418ms/step - loss: 0.2552 - mae: 0.4258 - val_loss: 0.0239 - val_mae: 0.1232
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step


In [12]:
!wget www.lamsade.dauphine.fr/~cazenave/sp500.csv

--2025-03-24 19:25:21--  http://www.lamsade.dauphine.fr/~cazenave/sp500.csv
Resolving www.lamsade.dauphine.fr (www.lamsade.dauphine.fr)... 193.48.71.250
Connecting to www.lamsade.dauphine.fr (www.lamsade.dauphine.fr)|193.48.71.250|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.lamsade.dauphine.fr/~cazenave/sp500.csv [following]
--2025-03-24 19:25:22--  https://www.lamsade.dauphine.fr/~cazenave/sp500.csv
Connecting to www.lamsade.dauphine.fr (www.lamsade.dauphine.fr)|193.48.71.250|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 48119 (47K) [text/csv]
Saving to: ‘sp500.csv’

sp500.csv           100%[===================>]  46.99K   197KB/s    in 0.2s    

2025-03-24 19:25:22 (197 KB/s) - ‘sp500.csv’ saved [48119/48119]



In [13]:
def normalise_windows(window_data):
  normalised_data = []
  for window in window_data:
    normalised_window = [((float(p) /
      float(window[0])) - 1) for p in window]
    normalised_data.append(normalised_window)
  return normalised_data

In [15]:
# Génération de texte
import keras
import numpy as np
path = keras.utils.get_file('nietzsche.txt', origin='https://s3.amazonaws.com/text-datasets/nietzsche.txt')
text = open(path).read().lower()
print('Corpus length:', len(text))

600901/600901 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Corpus length: 600893


In [17]:
# on considère que 60 caractères est une phrase
maxlen = 60
step = 3
sentences = []
next_chars = []
for i in range(0, len(text) - maxlen, step):
  sentences.append(text[i: i + maxlen])
  next_chars.append(text[i + maxlen])
print('Number of sequences:', len(sentences))
chars = sorted(list(set(text)))
print('Unique characters:', len(chars))
char_indices = dict((char, chars.index(char)) for char in chars)
print('Vectorization…')
x = np.zeros((len(sentences), maxlen, len(chars)), dtype=np.bool)
y = np.zeros((len(sentences), len(chars)), dtype=np.bool) # ens sortie la politique qui est le nombre de caractères uniques
for i, sentence in enumerate(sentences):
  for t, char in enumerate(sentence):
    x[i, t, char_indices[char]] = 1 # pour les entrées on met à 1 le caractère de chaque indice de la phrase
  y[i, char_indices[next_chars[i]]] = 1

Number of sequences: 200278
Unique characters: 57
Vectorization…


In [19]:
from keras import layers
model = keras.models.Sequential()
model.add(layers.LSTM(128, input_shape=(maxlen, len(chars))))
model.add(layers.Dense(len(chars), activation='softmax'))
optimizer = keras.optimizers.RMSprop(learning_rate=0.01)
model.compile(loss='categorical_crossentropy', optimizer=optimizer)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [20]:
def sample(preds, temperature=1.0):
  preds = np.asarray(preds).astype('float64')
  preds = np.log(preds) / temperature
  exp_preds = np.exp(preds)
  preds = exp_preds / np.sum(exp_preds)
  probas = np.random.multinomial(1, preds, 1)
  return np.argmax(probas)

In [ ]:
import random
import sys
for epoch in range(1, 60):
  model.fit(x, y, batch_size=128, epochs=1)
  print('epoch', epoch)
  start_index = random.randint(0, len(text) - maxlen - 1)
  generated_text = text[start_index: start_index + maxlen]
  for temperature in [0.2, 0.5, 1.0, 1.2]:
    sys.stdout.write(generated_text)
    print('\n\n------ temperature:', temperature)
    for i in range(400):
      sampled = np.zeros((1, maxlen, len(chars)))
      for t, char in enumerate(generated_text):
        sampled[0, t, char_indices[char]] = 1.
      preds = model.predict(sampled, verbose=0)[0]
      next_index = sample(preds, temperature)
      next_char = chars[next_index]
      generated_text += next_char
      generated_text = generated_text[1:]
      sys.stdout.write(next_char)

1565/1565 ━━━━━━━━━━━━━━━━━━━━ 235s 149ms/step - loss: 2.3149
epoch 1
influence of methodism (and
more recently as the "salvation 

------ temperature: 0.2
of the strong the strong the grated the state of the stander the stander the stander the made the suppority of the strong the string the and the madied to the strangerity, who has the state of the strong of the stander the more of the stander the propention of the stander the stranger the standeration. the made that the still to himself the moral the stander the stander the greater the state of thhe moral the stander the stander the greater the state of th

------ temperature: 0.5
e person
the morality of the forgned
of the propert in the man
the scienticmed to the stall of the formed
of the mashess of the strenger, in who who hast perhaps with it is the consequence of the madists morality
with nother, of the stander the and the belief the medistical more of the inthird, and probleness and that the perhaps and modes of
morality, wit